In [ ]:
import pandas as pd
import os
import json
import re
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Load CSV

In [ ]:
df = pd.read_csv("high_clv_bookings_with_persona_clusters.csv")

In [ ]:
CLUSTER_COL = "persona_cluster"  # change if your column name differs
print(df.shape)
print(df[CLUSTER_COL].value_counts().sort_index())
df.head(3)

# Cluster summary

In [ ]:
def top_values(series, n=8):
    s = series.dropna().astype(str)
    if len(s) == 0:
        return []
    vc = s.value_counts().head(n)
    return [{"value": k, "count": int(v)} for k, v in vc.items()]

def summarize_cluster(dfc: pd.DataFrame, max_top=8):
    cols = [c for c in dfc.columns if c != CLUSTER_COL]
    numeric_cols = [c for c in cols if pd.api.types.is_numeric_dtype(dfc[c])]
    cat_cols = [c for c in cols if c not in numeric_cols]

    numeric = {}
    for c in numeric_cols:
        s = pd.to_numeric(dfc[c], errors="coerce").dropna()
        if s.empty:
            continue
        numeric[c] = {
            "mean": float(s.mean()),
            "median": float(s.median()),
            "p25": float(s.quantile(0.25)),
            "p75": float(s.quantile(0.75)),
        }

    categorical_top = {}
    for c in cat_cols:
        tv = top_values(dfc[c], n=max_top)
        if tv:
            categorical_top[c] = tv

    return {
        "rows": int(len(dfc)),
        "numeric": numeric,
        "categorical_top": categorical_top
    }

cluster_summaries = {}
for cluster_id, dfc in df.groupby(CLUSTER_COL):
    cluster_summaries[int(cluster_id)] = summarize_cluster(dfc, max_top=8)

# Quick check
list(cluster_summaries.keys()), cluster_summaries[min(cluster_summaries.keys())]["rows"]


# Persona prompt template

In [ ]:
def build_persona_prompt(cluster_id: int, summary: dict) -> str:
    return f"""
You are an expert in hotel customer analytics, CRM strategy, and behavioral segmentation.

Your task is to INFER a realistic, privacy-safe hotel guest persona based strictly on the cluster summary below.

This persona will be used in an academic thesis and MUST NOT reveal any private or identifying business information.

--------------------------------------------------
CRITICAL CONSTRAINTS (MANDATORY):
- Use ONLY the information contained in the cluster summary.
- Do NOT invent demographics, income levels, or motivations not supported by data.
- Do NOT mention:
  • hotel names
  • brand names
  • product names
  • room or package titles
  • internal codes or SKUs
- Refer to offerings ONLY in abstract terms (e.g., "premium room", "wellness-oriented stay").

--------------------------------------------------
LANGUAGE QUALITY RULES:
- Avoid vague phrases such as:
  "values comfort", "seeks quality", "enjoys convenience", "likes premium experiences".
- Every statement must be behaviorally grounded and defensible from data.
- The persona must feel clearly distinguishable from other clusters.

--------------------------------------------------
WHAT YOU MUST INFER:
- Booking behavior (timing, frequency, flexibility)
- Value drivers (what triggers booking or upgrades)
- Price sensitivity vs. experience sensitivity
- Typical stay and service preferences (ABSTRACTED)
- Risk factors for disengagement or churn

--------------------------------------------------
OUTPUT FORMAT (STRICT JSON ONLY — NO NESTED OBJECTS):

Return a JSON object with EXACTLY these keys:

1. persona_name
   - 2–6 words
   - Distinctive, professional, and non-generic

2. persona_description
   - 6–9 sentences
   - Third-person narrative
   - Explain HOW this guest behaves, DECIDES, and RESPONDS

3. persona_tactics
   - A list of 8–12 PLAIN TEXT STRINGS ONLY
   - Each item must be ONE actionable sentence
   - ❌ No JSON objects
   - ❌ No keys like "tactic" or "description"
   - ✅ Example of correct format:
     "Trigger a personalized email when the guest shows inactivity, highlighting a flexible return incentive aligned with their past stay patterns."

--------------------------------------------------
CLUSTER CONTEXT:
Cluster ID: {cluster_id}

Cluster Summary (JSON):
{json.dumps(summary, ensure_ascii=False)}
""".strip()

# Meta ads prompt 3 distinct Meta ads

In [ ]:
TITLE_MAX = 30
DESC_MAX  = 90

def build_meta_ads_prompt(persona_name: str, persona_description: str, persona_tactics: str) -> str:
    tactics_short = (persona_tactics or "").split("\n")
    tactics_short = [t.strip() for t in tactics_short if t.strip()][:6]

    context = {
        "persona_name": persona_name,
        "persona_description": persona_description,
        "persona_tactics_samples": tactics_short
    }

    return f"""
You are writing Meta (Facebook/Instagram) ad copy for hospitality.

Return EXACTLY 3 ad variants.
If you return fewer or more than 3, your output is invalid.

STRICT RULES:
- Titles: max 30 characters. Descriptions: max 90 characters.
- Each of the 3 variants must be meaningfully distinct (different intent angle).
- Do NOT mention hotel names, brand names, product names, room/package titles.
- Use only abstract language (e.g., "a flexible stay", "a curated getaway").
- No hashtags. No markdown. No quotes. No emojis. No line breaks.

MANDATORY ANGLES:
Ad 1: Emotional / experiential
Ad 2: Practical / value-driven
Ad 3: Action / urgency (no false claims)

OUTPUT MUST BE VALID JSON ONLY in this exact structure:
{{
  "ads": [
    {{"title": "...", "description": "..."}},
    {{"title": "...", "description": "..."}},
    {{"title": "...", "description": "..."}}
  ]
}}

Persona context (JSON):
{json.dumps(context, ensure_ascii=False)}
""".strip()


# Parsers + enforcement (handles weird outputs safely)

In [ ]:
def extract_json(text: str) -> dict:
    t = (text or "").strip()

    # 1) Direct parse
    try:
        return json.loads(t)
    except Exception:
        pass

    # 2) Fallback: grab the biggest {...} block
    m = re.search(r"\{.*\}", t, flags=re.DOTALL)
    if not m:
        raise ValueError("No JSON object found in model output.")

    return json.loads(m.group(0))


def repair_to_json(call_llm, bad_text: str, schema_hint: str) -> dict:
    repair_prompt = f"""
Convert the following into VALID JSON only.

Rules:
- Output MUST be valid JSON.
- No extra text, no markdown.
- Must follow this schema exactly:
{schema_hint}

Text to fix:
{bad_text}
""".strip()

    fixed = call_llm(repair_prompt)
    return extract_json(fixed)


def normalize_tactics(tactics):
    cleaned = []
    if isinstance(tactics, list):
        for t in tactics:
            if isinstance(t, dict):
                if "description" in t:
                    cleaned.append(str(t["description"]).strip())
                else:
                    cleaned.append(" ".join(map(str, t.values())).strip())
            else:
                cleaned.append(str(t).strip())
    else:
        cleaned.append(str(tactics).strip())
    cleaned = [t for t in cleaned if len(t) > 20]
    return cleaned

def _clean_text(s: str) -> str:
    if s is None: return ""
    s = str(s).replace("\n", " ").replace("\r", " ")
    # no quotes
    s = s.replace('"', "").replace("'", "").replace("“", "").replace("”", "").replace("’", "")
    # no hashtags
    s = re.sub(r"#\w+", "", s)
    # no markdown-ish chars
    s = s.replace("*", "").replace("_", "").replace("`", "")
    s = re.sub(r"\s+", " ", s).strip()
    return s

def _truncate(s: str, max_len: int) -> str:
    s = _clean_text(s)
    if len(s) <= max_len: return s
    s = s[:max_len].rstrip(" ,.;:-")
    return s

def enforce_meta_rules(title: str, desc: str):
    title = _truncate(title, TITLE_MAX)
    desc  = _truncate(desc,  DESC_MAX)
    if not title:
        title = _truncate("Plan your next escape", TITLE_MAX)
    if not desc:
        desc = _truncate("Flexible stays and curated moments to help you truly recharge.", DESC_MAX)
    return title, desc

# Provider adapters (Gemini / OpenAI / Claude / Grok / DeepSeek)

## Gemini adapter (google-generativeai) - gemini-2.5-pro and gemini-3-pro

In [ ]:
import google.generativeai as genai

def make_gemini_caller(api_key: str, model: str):
    genai.configure(api_key=api_key)
    gm = genai.GenerativeModel(model)
    def _call(prompt: str) -> str:
        return gm.generate_content(prompt).text
    return _call

## OpenAI-compatible adapter (OpenAI GPT-5, xAI Grok, DeepSeek) - gpt-5, grok-4-1-fast-reasoning & deepseek-chat

In [ ]:
from openai import OpenAI

def make_openai_compat_caller(api_key: str, model: str, base_url: str | None = None, use_responses: bool = False):
    client = OpenAI(api_key=api_key, base_url=base_url) if base_url else OpenAI(api_key=api_key)

    def _call(prompt: str) -> str:
        if use_responses:
            resp = client.responses.create(
                model=model,
                input=prompt
            )
            return resp.output_text
        else:
            kwargs = dict(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7,
                max_tokens=900,   # <<< ADD THIS
            )

            # Optional: try to force JSON mode if provider supports it
            # If xAI rejects it, we silently fall back.
            try:
                kwargs["response_format"] = {"type": "json_object"}
            except Exception:
                pass

            resp = client.chat.completions.create(**kwargs)
            return resp.choices[0].message.content

    return _call

## Claude - claude-4.5-sonnet

In [ ]:
from anthropic import Anthropic

def make_claude_caller(api_key: str, model: str):
    client = Anthropic(api_key=api_key)
    def _call(prompt: str) -> str:
        msg = client.messages.create(
            model=model,
            max_tokens=1400,
            messages=[{"role": "user", "content": prompt}]
        )
        # Anthropic returns a list of content blocks
        return "".join([b.text for b in msg.content if hasattr(b, "text")])
    return _call

# Defining 5 providers + keys

In [ ]:
os.environ["GEMINI_API_KEY"]     = "API-KEY"
os.environ["OPENAI_API_KEY"]     = "API-KEY"
os.environ["ANTHROPIC_API_KEY"]  = "API-KEY"
os.environ["XAI_API_KEY"]        = "API-KEY"
os.environ["DEEPSEEK_API_KEY"]   = "API-KEY"

In [ ]:
PROVIDERS = [
    {
        "label": "GPT_5",
        "caller": make_openai_compat_caller(
            api_key=os.environ["OPENAI_API_KEY"],
            model="gpt-5",
            use_responses=True
        )
    },
     {
         "label": "Claude_4.5_Sonnet",
        "caller": make_claude_caller(
             api_key=os.environ["ANTHROPIC_API_KEY"],
             model="claude-sonnet-4-5-20250929"
        )
     },
          {
     "label": "Claude_Opus_4.6",
     "caller": make_claude_caller(
         api_key=os.environ["ANTHROPIC_API_KEY"],
         model="claude-opus-4-6"
     )
 },
    
     {
         "label": "Grok_4.1_fast_reasoning",
         "caller": make_openai_compat_caller(
             api_key=os.environ["XAI_API_KEY"],
             model="grok-4-1-fast-reasoning",
             base_url="https://api.x.ai/v1"
         )
     },
     {
         "label": "DeepSeek_chat",
         "caller": make_openai_compat_caller(
             api_key=os.environ["DEEPSEEK_API_KEY"],
             model="deepseek-chat",
             base_url="https://api.deepseek.com/v1"
         )
     },
     {
         "label": "Gemini_2.5_flash",
         "caller": make_gemini_caller(
             api_key=os.environ["GEMINI_API_KEY"],
            model="gemini-2.5-flash-preview" 
         )
     },
          {
         "label": "Gemini_2.5_pro",
         "caller": make_gemini_caller(
             api_key=os.environ["GEMINI_API_KEY"],
            model="gemini-2.5-pro-preview" 
         )
     },
     {
         "label": "Gemini_3_pro",
         "caller": make_gemini_caller(
             api_key=os.environ["GEMINI_API_KEY"],
            model="gemini-3-pro-preview" 
        )
     },
     {
         "label": "Gemini_3.1_pro",
         "caller": make_gemini_caller(
             api_key=os.environ["GEMINI_API_KEY"],
             model="gemini-3.1-pro-preview" 
         )
     },
]

# Generating personas + ads, then saves CSV

In [ ]:
def run_provider(label: str, call_llm):
    # 1) personas
    persona_rows = []
    for cluster_id in sorted(cluster_summaries.keys()):
        p_prompt = build_persona_prompt(cluster_id, cluster_summaries[cluster_id])
        raw = call_llm(p_prompt)

        persona_schema = """{
        "persona_name": "string",
        "persona_description": "string",
        "persona_tactics": ["string"]
        }"""

        try:
            data = extract_json(raw)
        except Exception:
            data = repair_to_json(call_llm, raw, persona_schema)


        tactics_list = normalize_tactics(data.get("persona_tactics", []))
        persona_rows.append({
            "persona_cluster": cluster_id,
            "cluster_rows": cluster_summaries[cluster_id]["rows"],
            "persona_name": str(data.get("persona_name", "")).strip(),
            "persona_description": str(data.get("persona_description", "")).strip(),
            "persona_tactics": "\n".join(tactics_list)
        })

    personas_df = pd.DataFrame(persona_rows).sort_values("persona_cluster").reset_index(drop=True)

    # 2) ads
    ad_rows = []
    for _, r in personas_df.iterrows():
        a_prompt = build_meta_ads_prompt(
            r["persona_name"],
            r["persona_description"],
            r["persona_tactics"],
        )
        raw = call_llm(a_prompt)

        ads_schema = """{
        "ads": [
            {"title": "string", "description": "string"},
            {"title": "string", "description": "string"},
            {"title": "string", "description": "string"}
        ]
        }"""

        try:
            data = extract_json(raw)
        except Exception:
            data = repair_to_json(call_llm, raw, ads_schema)

        ads = data.get("ads", [])
        if not isinstance(ads, list) or len(ads) != 3:
            raise ValueError(f"{label}: expected exactly 3 ads, got: {ads}")

        fixed = []
        for ad in ads:
            t = ad.get("title", "") if isinstance(ad, dict) else ""
            d = ad.get("description", "") if isinstance(ad, dict) else ""
            t, d = enforce_meta_rules(t, d)
            fixed.append((t, d))

        ad_rows.append({
            "ad_title_1": fixed[0][0], "ad_desc_1": fixed[0][1],
            "ad_title_2": fixed[1][0], "ad_desc_2": fixed[1][1],
            "ad_title_3": fixed[2][0], "ad_desc_3": fixed[2][1],
        })

    ads_df = pd.DataFrame(ad_rows)
    final_df = pd.concat([personas_df, ads_df], axis=1)

    # 3) save
    out_path = f"personas_metaads_{label}.csv"
    final_df.to_csv(out_path, index=False)
    return out_path, final_df

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 10

# v4 scores
models = ['Claude\nOpus 4.6', 'Gemini\n3 Pro', 'Gemini\n3.1 Pro', 'GPT-5',
          'Claude\n4.5 Sonnet', 'DeepSeek\nChat', 'Grok\n4.1 Fast']
description = [9, 9, 8, 8, 7, 5, 6]
tactics     = [10, 9, 8, 9, 8, 6, 6]
meta_ads    = [7, 9, 9, 8, 8, 4, 2]
images      = [9, 7, 8, 7, 7, 8, 8]

x = np.arange(len(models))
width = 0.2

fig, ax = plt.subplots(figsize=(11, 5))

ax.bar(x - 1.5*width, description, width, label='Description', color='#4a7ba6', edgecolor='black', linewidth=0.4)
ax.bar(x - 0.5*width, tactics,     width, label='Tactics',     color='#6b8e76', edgecolor='black', linewidth=0.4)
ax.bar(x + 0.5*width, meta_ads,    width, label='Meta Ads',    color='#c97a3a', edgecolor='black', linewidth=0.4)
ax.bar(x + 1.5*width, images,      width, label='Images',      color='#8e6ba6', edgecolor='black', linewidth=0.4)

ax.set_xlabel('Model')
ax.set_ylabel('Score (max 10)')
ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=9)
ax.set_ylim(0, 11)
ax.set_yticks(range(0, 11, 2))
ax.legend(frameon=False, loc='upper right', ncol=4, fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle=':', alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('figure_8_llm_scores_v4.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 10

versions = ['v1', 'v2', 'v3', 'v4']

# Total scores per model per version
# v3 is out of 30 (image dimension not assessed); v1, v2, v4 are out of 40
gpt5         = [29, 31, 25, 32]
claude_45    = [27, 27, 23, 30]
grok         = [23, 24, 14, 22]
deepseek     = [23, 21, 15, 23]

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(versions, gpt5,      marker='o', linewidth=2, color='#4a7ba6', label='GPT-5')
ax.plot(versions, claude_45, marker='s', linewidth=2, color='#6b8e76', label='Claude 4.5 Sonnet')
ax.plot(versions, grok,      marker='^', linewidth=2, color='#c97a3a', label='Grok 4.1 Fast')
ax.plot(versions, deepseek,  marker='d', linewidth=2, color='#8e6ba6', label='DeepSeek Chat')

# Annotate v3 specially since it has different max
ax.annotate('v3: max 30\n(no image dimension)',
            xy=(2, 14), xytext=(2, 6),
            fontsize=8, ha='center', color='#555',
            arrowprops=dict(arrowstyle='->', color='#888', lw=0.8))

ax.set_xlabel('Pipeline version')
ax.set_ylabel('Total score')
ax.set_ylim(0, 42)
ax.set_yticks(range(0, 41, 5))
ax.legend(frameon=False, loc='lower left', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(axis='y', linestyle=':', alpha=0.4)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('figure_9_score_evolution.png', dpi=300, bbox_inches='tight')
plt.show()

# Run all 5 providers and produce 5 CSVs

In [ ]:
outputs = {}

for p in PROVIDERS:
    label = p["label"]
    print("Running:", label)
    out_path, out_df = run_provider(label, p["caller"])
    outputs[label] = out_path
    print("Saved:", out_path)

outputs